In [ ]:
%python
# ============================================================
# CONFIGURATION — update these two values before running
# ============================================================
catalog = "serverless_stable_ysmqdz_catalog"  # e.g. "main"
schema  = "abac_test"   # e.g. "abac_demo"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
print(f"✓ Using: {catalog}.{schema}")
print(f"✓ Running as: {current_user}")


In [ ]:
# ── Cleanup: drop all policies from previous runs to avoid conflicts ──
_all_policies = ['abac_hc_emergency_access', 'abac_hc_regional_access_deny', 'abac_hc_regional_access_east', 'abac_hc_regional_access_north', 'abac_hc_regional_access_south', 'abac_hc_regional_access_west', 'fraud_detection_filter', 'region_row_filter_apac_policy', 'region_row_filter_catalog_apac', 'region_row_filter_catalog_deny', 'region_row_filter_catalog_eu', 'region_row_filter_catalog_us', 'region_row_filter_deny_policy', 'region_row_filter_eu_policy', 'region_row_filter_us_policy', 'flexible_mask_metadata', 'hipaa_mask_contact', 'hipaa_mask_dob', 'hipaa_mask_ids', 'hipaa_mask_name', 'mask_ssn_policy', 'pci_mask_credit_card', 'pci_mask_income', 'pci_mask_ssn', 'pseudo_email_policy', 'redact_other_pii', 'redact_pii', 'clearance_access_policy', 'pii_mask_email', 'pii_mask_name', 'user_access_filter', 'pci_pending_lockdown', 'pci_reviewed_card', 'pci_reviewed_cvv', 'pci_reviewed_name', 'review_complete_classified', 'review_complete_policy', 'review_pending_policy', 'mask_confidential_finance', 'mask_confidential_hr', 'mask_confidential_marketing', 'mask_internal_finance', 'mask_internal_hr', 'mask_internal_marketing', 'telco_billing_confidential', 'telco_billing_restricted', 'telco_cpni_internal', 'telco_cpni_restricted', 'telco_identity_confidential', 'telco_identity_internal', 'telco_identity_restricted', 'telco_service_row_filter_bundle', 'telco_service_row_filter_data', 'telco_service_row_filter_deny', 'telco_service_row_filter_voice']

_dropped, _skipped = 0, 0
for _p in _all_policies:
    try:
        spark.sql(f"DROP POLICY {_p} ON SCHEMA {catalog}.{schema}")
        _dropped += 1
    except Exception:
        _skipped += 1

print(f"Cleanup done — dropped: {_dropped}, not found: {_skipped}")


## Who Sees What — Group Membership Guide

Add `abhishekpratap.singh@databricks.com` to one of these account-level groups (and remove from `abac_tut_admins`) to see the filters in action.

### `customers` table (regions: us / eu / apac)
| Your Group | Rows You See | Why |
|---|---|---|
| `abac_tut_us_team` | Rows 1, 2, 3, 10 | `region_filter_us` policy applied to your group |
| `abac_tut_eu_team` | Rows 4, 5, 6, 11 | `region_filter_eu` policy applied to your group |
| `abac_tut_apac_team` | Rows 7, 8, 9, 12 | `region_filter_apac` policy applied to your group |
| `abac_tut_admins` | All 12 rows | Exempt via EXCEPT in deny policy |
| *(no group)* | 0 rows | Deny-all policy applies |

### `hospital_visits` table (regions: north / south / east / west)
| Your Group | Rows You See | Why |
|---|---|---|
| `abac_tut_us_team` | Visits 1, 3 (north) + 5, 8 (west) | Two policies map north+west to us_team |
| `abac_tut_eu_team` | Visits 2, 6 (south) | south → eu_team |
| `abac_tut_apac_team` | Visits 4, 7 (east) | east → apac_team |
| `abac_tut_admins` | All 8 visits | Exempt |
| *(no group)* | 0 rows | Deny-all policy applies |

### `transactions` table (fraud_flag filter)
| Your Group | Rows You See | Why |
|---|---|---|
| `abac_tut_finance_team` | All 10 transactions | Exempt from fraud filter (fraud analysts) |
| `abac_tut_hr_team` | All 10 transactions | Exempt from fraud filter (compliance) |
| `abac_tut_admins` | All 10 transactions | Exempt |
| *(anyone else)* | Only fraud=TRUE rows (1003, 1004, 1007, 1009) | `fraud_row_filter` applied |


# Tutorial 1: Row Filtering with ABAC

This tutorial demonstrates how to use ABAC to filter table rows based on governed tags and group membership.

**Requirements:**

- Databricks Runtime 16.4+ or serverless compute
- A Unity Catalog-enabled workspace
- Governed tags created via the Catalog Explorer UI (see Setup section)
- `CREATE SCHEMA`, `CREATE TABLE`, `CREATE FUNCTION` privileges on the target catalog
- Ownership or `MANAGE` privilege on the catalog/schema to create policies
- `APPLY TAG` and `ASSIGN` privilege on governed tags
- Account groups: `abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_admins`

## Setup

> **Note:** This tutorial uses `{catalog}` as the catalog name. Replace with your own catalog if needed.

### Create account groups

This tutorial references: `abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_admins`

Account-level groups must be created in the **Account Console** (Admin > Groups), not via SQL.

> If these groups already exist, skip this step.

### Create governed tags

Create the following key-only governed tag in the Catalog Explorer UI (**Catalog** > **Governed Tags** > **Create governed tag**):

| Tag Key | Allowed Values |
|---------|---------------|
| `abac_tut_region` | *(none — key-only tag)* |

> If this tag already exists, skip this step.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.customers")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.customers (
  id INT, name STRING, email STRING, region STRING, revenue DOUBLE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.customers VALUES
  (1,  'Alice Johnson',    'alice@acme.com',       'us',   50000.00),
  (2,  'Bob Smith',        'bob@acme.com',         'us',   120000.00),
  (3,  'Carol White',      'carol@acme.com',       'us',   500000.00),
  (4,  'Hans Mueller',     'hans@firma.de',        'eu',   45000.00),
  (5,  'Sophie Dubois',    'sophie@societe.fr',    'eu',   95000.00),
  (6,  'Lars Eriksson',    'lars@foretag.se',      'eu',   320000.00),
  (7,  'Yuki Tanaka',      'yuki@kaisha.jp',       'apac', 38000.00),
  (8,  'Wei Chen',         'wei@gongsi.cn',        'apac', 110000.00),
  (9,  'Priya Sharma',     'priya@company.in',     'apac', 275000.00),
  (10, 'James Wilson',     'james@corp.com',       'us',   88000.00),
  (11, 'Maria Garcia',     'maria@empresa.es',     'eu',   62000.00),
  (12, 'Kenji Watanabe',   'kenji@kigyou.jp',      'apac', 73000.00)
""")


In [ ]:
spark.sql(f"""
-- Grant base access so group members can query the tables.
-- ABAC policies handle the fine-grained row/column filtering on top of these grants.
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
spark.sql(f"""
-- Tag the region COLUMN so MATCH COLUMNS can find it
ALTER TABLE {catalog}.{schema}.customers ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")


## Step 1: Create the Row Filter UDF

This UDF receives a row's region value and returns TRUE if the current user belongs to the matching regional group.

In [ ]:
spark.sql(f"""
-- UDF: pure data logic, no identity checks
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_us(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'us'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_eu(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'eu'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_apac(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'apac'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_deny(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN FALSE
""")

## Step 2: Create the Row Filter Policy

The policy uses `MATCH COLUMNS` to find columns tagged `abac_tut_region` and passes their value to the UDF via `USING COLUMNS`.

In [ ]:
spark.sql(f"""
-- Policies: group membership defined here, not in UDFs
CREATE OR REPLACE POLICY region_row_filter_us_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_us
TO abac_tut_us_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_eu_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_eu
TO abac_tut_eu_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_apac_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_apac
TO abac_tut_apac_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
-- Default deny for users not in any regional group (excluding admins)
CREATE OR REPLACE POLICY region_row_filter_deny_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_deny
TO `account users` EXCEPT abac_tut_us_team, abac_tut_eu_team, abac_tut_apac_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

## Step 3: EXCEPT Clause

The `EXCEPT` clause exempts specific groups from the policy. Here we exempt `abac_tut_admins` so they see all rows.

### What to expect from this query

This runs **after schema-level region policies** are created.

| Your Group | Rows returned | Example names |
|---|---|---|
| `abac_tut_us_team` | 4 rows (region = us) | Alice Johnson, Bob Smith, Carol White, James Wilson |
| `abac_tut_eu_team` | 4 rows (region = eu) | Hans Mueller, Sophie Dubois, Lars Eriksson, Maria Garcia |
| `abac_tut_apac_team` | 4 rows (region = apac) | Yuki Tanaka, Wei Chen, Priya Sharma, Kenji Watanabe |
| `abac_tut_admins` | **12 rows** | All customers |
| *(no group)* | **0 rows** | Deny-all policy |


In [ ]:
print(f"Running as: {current_user}")
spark.sql(f"""SELECT * FROM {catalog}.{schema}.customers ORDER BY region, id""").show(truncate=False)


## Step 4: Hierarchical Scope

Policies can be set at catalog, schema, or table level. A catalog-level policy applies to all tables in all schemas. Here we promote the region filter to catalog scope and show it applying across schemas.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.orders")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.orders (
  order_id INT, customer_name STRING, region STRING, amount DOUBLE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.orders VALUES
  (101, 'Alice Johnson', 'us', 2500.00), (102, 'Hans Mueller', 'eu', 1800.00),
  (103, 'Yuki Tanaka', 'apac', 3200.00), (104, 'Bob Smith', 'us', 4100.00),
  (105, 'Sophie Dubois', 'eu', 2900.00)
""")

spark.sql(f"""
-- Tag the region column on the orders table too
ALTER TABLE {catalog}.{schema}.orders ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")


In [ ]:
# No additional grants needed — orders is in the same schema as customers


In [ ]:
spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_catalog_us
ON CATALOG {catalog}
ROW FILTER {catalog}.{schema}.region_filter_us
TO abac_tut_us_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_catalog_eu
ON CATALOG {catalog}
ROW FILTER {catalog}.{schema}.region_filter_eu
TO abac_tut_eu_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_catalog_apac
ON CATALOG {catalog}
ROW FILTER {catalog}.{schema}.region_filter_apac
TO abac_tut_apac_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_catalog_deny
ON CATALOG {catalog}
ROW FILTER {catalog}.{schema}.region_filter_deny
TO `account users` EXCEPT abac_tut_us_team, abac_tut_eu_team, abac_tut_apac_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")


### What to expect — catalog-level policy across both tables

The catalog-level policy applies the **same regional filter to every tagged table** in the catalog — you do not need to set policies per table.

| Your Group | customers rows | orders rows |
|---|---|---|
| `abac_tut_us_team` | US rows (1,2,3,10) | Orders 101, 104 |
| `abac_tut_eu_team` | EU rows (4,5,6,11) | Orders 102, 105 |
| `abac_tut_apac_team` | APAC rows (7,8,9,12) | Order 103 |
| `abac_tut_admins` | All 12 | All 5 orders |


In [ ]:
print(f"Running as: {current_user}")
print("--- customers (catalog-level policy) ---")
spark.sql(f"""SELECT id, name, region FROM {catalog}.{schema}.customers ORDER BY region, id""").show(truncate=False)
print("--- orders (same catalog-level policy auto-applies) ---")
spark.sql(f"""SELECT order_id, customer_name, region, amount FROM {catalog}.{schema}.orders ORDER BY region""").show(truncate=False)


## Expected Results

| User Group | Customers Visible | Orders Visible |
|------------|-------------------|----------------|
| `abac_tut_us_team` | US rows (1, 2, 3, 10) | US orders (101, 104) |
| `abac_tut_eu_team` | EU rows (4, 5, 6, 11) | EU orders (102, 105) |
| `abac_tut_apac_team` | APAC rows (7, 8, 9, 12) | APAC orders (103) |
| `abac_tut_admins` | All 12 rows | All 5 orders |

### Delete governed tags

If you no longer need it, delete `abac_tut_region` from the Catalog Explorer UI (**Governed Tags** section).

---
## Industry Examples

> **Instructions:** Replace `{catalog}` with your Unity Catalog catalog name and `{schema}` with your target schema name before running.
>
> Groups referenced in this section (`abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_us_team`, `abac_tut_admins`, `abac_tut_admins`, `abac_tut_finance_team`, `abac_tut_hr_team`) must be created in the **Account Console** by an admin.
>
> Governed tags `abac_tut_region` and `abac_tut_emergency` (key-only) must be created in the Catalog Explorer UI.

## Groups Used in Industry Examples

The industry examples below reuse the same account groups from the core tutorial:

| Group | Used As |
|-------|---------|
| `abac_tut_us_team` | Regional (US/North/Voice) |
| `abac_tut_eu_team` | Regional (EU/South/Data) |
| `abac_tut_apac_team` | Regional (APAC/East/Bundle) |
| `abac_tut_admins` | Admin exemption (all policies) |
| `abac_tut_hr_team` | HR domain / Identity owners |
| `abac_tut_finance_team` | Finance domain / Billing / Fraud analysts |
| `abac_tut_marketing_team` | Marketing domain / CPNI owners |

> These groups are managed by your account admin. No group creation SQL is needed here.

## Healthcare — Time-Based and Role-Based Row Filtering

Hospitals must restrict access to patient visit data based on staff region and role. On-call staff need emergency records even outside business hours, while standard staff only access their regional patients during business hours.

**Compliance context:** HIPAA requires minimum necessary access — staff should only see patient records relevant to their care responsibilities.

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
# Drop general regional policies before healthcare section to avoid multiple row filter conflicts
for _p in ['region_row_filter_us_policy','region_row_filter_eu_policy','region_row_filter_apac_policy',
           'region_row_filter_deny_policy','region_row_filter_catalog_us','region_row_filter_catalog_eu',
           'region_row_filter_catalog_apac','region_row_filter_catalog_deny']:
    try: spark.sql(f'DROP POLICY {_p} ON SCHEMA {catalog}.{schema}')
    except: pass
    try: spark.sql(f'DROP POLICY {_p} ON CATALOG {catalog}')
    except: pass
print('Regional policies cleared')


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.hospital_visits")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.hospital_visits (
  visit_id INT,
  patient_id INT,
  patient_name STRING,
  region STRING,
  department STRING,
  visit_date DATE,
  is_emergency BOOLEAN,
  attending_physician STRING,
  diagnosis_code STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.hospital_visits VALUES
  (1, 101, 'Alice Brown',    'north', 'cardiology', '2026-01-15', FALSE, 'Dr. Smith', 'I21.0'),
  (2, 102, 'Bob Davis',      'south', 'neurology',  '2026-01-16', TRUE,  'Dr. Jones', 'G35'),
  (3, 103, 'Carol Evans',    'north', 'oncology',   '2026-01-17', FALSE, 'Dr. Lee',   'C34.1'),
  (4, 104, 'David Garcia',   'east',  'cardiology', '2026-01-18', TRUE,  'Dr. Patel', 'I50.9'),
  (5, 105, 'Eva Harris',     'west',  'neurology',  '2026-01-19', FALSE, 'Dr. Kim',   'G43.9'),
  (6, 106, 'Frank Iyer',     'south', 'oncology',   '2026-01-20', FALSE, 'Dr. Chen',  'C50.9'),
  (7, 107, 'Grace Johnson',  'east',  'cardiology', '2026-01-21', TRUE,  'Dr. Wong',  'I10'),
  (8, 108, 'Henry Kumar',    'west',  'neurology',  '2026-01-22', FALSE, 'Dr. Gupta', 'G20')
""")


In [ ]:
spark.sql(f"""
-- Tag region column for ABAC policy matching
ALTER TABLE {catalog}.{schema}.hospital_visits ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")

spark.sql(f"""
-- Tag is_emergency column for emergency access policy
ALTER TABLE {catalog}.{schema}.hospital_visits ALTER COLUMN is_emergency SET TAGS ('abac_tut_emergency' = '')
""")


### Healthcare Filter 1: Business Hours Access

Standard staff can only access patient records during business hours (7AM–7PM, Monday–Friday). This is enforced server-side by the UDF using `CURRENT_TIMESTAMP()`.

In [ ]:
spark.sql(f"""
-- Business hours filter: 7AM-7PM Monday-Friday
-- DAYOFWEEK: 1=Sunday, 2=Monday, ..., 6=Friday, 7=Saturday
CREATE OR REPLACE FUNCTION {catalog}.{schema}.business_hours_filter(dummy STRING)
RETURNS BOOLEAN
RETURN (HOUR(CURRENT_TIMESTAMP()) BETWEEN 7 AND 19)
  AND (DAYOFWEEK(CURRENT_DATE()) BETWEEN 2 AND 6)
""")


### Healthcare Filter 2: Emergency Access Override

On-call staff (members of `abac_tut_admins`) can always access emergency records regardless of business hours. Non-senior staff only see emergency visits.

In [ ]:
spark.sql(f"""
-- Emergency access: show only emergency=TRUE rows
-- The EXCEPT abac_tut_admins in the policy handles admin exemption
CREATE OR REPLACE FUNCTION {catalog}.{schema}.emergency_access_filter(is_emergency BOOLEAN)
RETURNS BOOLEAN
DETERMINISTIC
RETURN is_emergency = TRUE
-- Policy uses EXCEPT abac_tut_admins to exempt admins
""")

### Healthcare Filter 3: Regional Access Control

Each regional team (north, south, east, west) can only see patients from their own region. Admins are exempted via the EXCEPT clause.

In [ ]:
spark.sql(f"""
-- UDFs: pure data logic, no identity checks
CREATE OR REPLACE FUNCTION {catalog}.{schema}.abac_hc_region_filter_north(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'north'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.abac_hc_region_filter_south(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'south'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.abac_hc_region_filter_east(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'east'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.abac_hc_region_filter_west(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'west'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.abac_hc_region_filter_deny(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN FALSE
""")

### Healthcare Policy: Schema-Level Regional Row Filter

Apply the regional filter at schema level. The `EXCEPT abac_tut_admins` clause ensures hospital administrators see all patient records across all regions.

In [ ]:
spark.sql(f"""
-- Policies: group membership defined here, not in UDFs
-- abac_tut_us_team covers both north and west regions
CREATE OR REPLACE POLICY abac_hc_regional_access_north
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.abac_hc_region_filter_north
TO abac_tut_us_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY abac_hc_regional_access_west
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.abac_hc_region_filter_west
TO abac_tut_us_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY abac_hc_regional_access_south
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.abac_hc_region_filter_south
TO abac_tut_eu_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY abac_hc_regional_access_east
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.abac_hc_region_filter_east
TO abac_tut_apac_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
-- Default deny for users not in any regional group (excluding admins)
CREATE OR REPLACE POLICY abac_hc_regional_access_deny
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.abac_hc_region_filter_deny
TO `account users` EXCEPT abac_tut_us_team, abac_tut_eu_team, abac_tut_apac_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

### What to expect from this query

Regional filter applied to `hospital_visits`. `abac_tut_us_team` maps to **north + west** (two separate policies, OR'd together).

| Your Group | Visit IDs | Regions |
|---|---|---|
| `abac_tut_us_team` | 1, 3, 5, 8 | north (1,3) + west (5,8) |
| `abac_tut_eu_team` | 2, 6 | south |
| `abac_tut_apac_team` | 4, 7 | east |
| `abac_tut_admins` | All 8 | all regions |
| *(no group)* | 0 rows | deny policy |


In [ ]:
print(f"Running as: {current_user}")
spark.sql(f"""SELECT visit_id, patient_name, region, department, is_emergency FROM {catalog}.{schema}.hospital_visits ORDER BY region, visit_id""").show(truncate=False)


### Healthcare Policy: Emergency Access Filter

Apply the emergency access filter using the `abac_tut_emergency` tag on the `is_emergency` column.

In [ ]:
spark.sql(f"""
-- Emergency access row filter policy
-- Non-senior staff can only see emergency visits (is_emergency = TRUE)
CREATE OR REPLACE POLICY abac_hc_emergency_access
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.emergency_access_filter
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_emergency') AS emerg_col
USING COLUMNS (emerg_col)
""")


**Expected results:**

| Group | Visible Visits |
|-------|----------------|
| `abac_tut_us_team` | Visits 1, 3 (north region, both policies apply — regional first) |
| `abac_tut_admins` | Emergency visits 2, 4, 7 plus any regional access |
| `abac_tut_admins` | All 8 visits (exempt from both policies) |
| Non-member | No visits (fail-closed) |

---
## Financial Services — Fraud Detection Row Filtering

Fraud analysts need to see all flagged transactions across regions. Compliance officers need visibility into high-value transactions regardless of fraud status. Regional operations teams only see their own region's transactions.

**Compliance context:** SOX and PCI-DSS require audit trails and restricted access to transaction data. Only authorized roles should have access to transaction records.

In [ ]:
# Drop healthcare policies before financial section to avoid multiple row filter conflicts
for _p in ['abac_hc_emergency_access','abac_hc_regional_access_deny','abac_hc_regional_access_east',
           'abac_hc_regional_access_north','abac_hc_regional_access_south','abac_hc_regional_access_west']:
    try: spark.sql(f'DROP POLICY {_p} ON SCHEMA {catalog}.{schema}')
    except: pass
print('Healthcare policies cleared')


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.transactions")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.transactions (
  txn_id      INT,
  account_id  STRING,
  customer_name STRING,
  region      STRING,
  amount      DOUBLE,
  currency    STRING,
  txn_type    STRING,
  fraud_flag  BOOLEAN,
  risk_score  INT,
  txn_date    DATE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.transactions VALUES
  (1001, 'ACC-001', 'Alice Brown',   'us_east', 450.00,    'USD', 'purchase', FALSE, 12,  '2026-01-10'),
  (1002, 'ACC-002', 'Bob Davis',     'us_west', 12500.00,  'USD', 'wire',     FALSE, 35,  '2026-01-11'),
  (1003, 'ACC-003', 'Carol Evans',   'eu',      880.50,    'EUR', 'purchase', TRUE,  91,  '2026-01-12'),
  (1004, 'ACC-004', 'David Garcia',  'us_east', 25000.00,  'USD', 'wire',     TRUE,  88,  '2026-01-13'),
  (1005, 'ACC-005', 'Eva Harris',    'apac',    320.00,    'SGD', 'purchase', FALSE, 8,   '2026-01-14'),
  (1006, 'ACC-006', 'Frank Iyer',    'eu',      75000.00,  'EUR', 'transfer', FALSE, 42,  '2026-01-15'),
  (1007, 'ACC-007', 'Grace Johnson', 'us_west', 1200.00,   'USD', 'purchase', TRUE,  79,  '2026-01-16'),
  (1008, 'ACC-008', 'Henry Kumar',   'apac',    18000.00,  'USD', 'wire',     FALSE, 55,  '2026-01-17'),
  (1009, 'ACC-009', 'Iris Lee',      'us_east', 5.99,      'USD', 'purchase', TRUE,  95,  '2026-01-18'),
  (1010, 'ACC-010', 'James Miller',  'eu',      150000.00, 'EUR', 'transfer', FALSE, 61,  '2026-01-19')
""")


In [ ]:
spark.sql(f"""
-- Tag region and fraud_flag columns for ABAC policy matching
ALTER TABLE {catalog}.{schema}.transactions ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.transactions ALTER COLUMN fraud_flag SET TAGS ('abac_tut_emergency' = '')
""")


In [ ]:
spark.sql(f"""
-- Fraud analysts: only see fraud-flagged transactions
-- The EXCEPT abac_tut_hr_team, abac_tut_admins in the policy handles exemptions
CREATE OR REPLACE FUNCTION {catalog}.{schema}.fraud_row_filter(fraud_flag BOOLEAN)
RETURNS BOOLEAN
DETERMINISTIC
RETURN fraud_flag = TRUE
-- Policy uses EXCEPT abac_tut_hr_team, abac_tut_admins to exempt those groups
""")

In [ ]:
spark.sql(f"""
-- High-value transaction filter: compliance team sees transactions > $10,000
-- The EXCEPT clause in the policy handles abac_tut_hr_team exemption
CREATE OR REPLACE FUNCTION {catalog}.{schema}.high_value_filter(amount DOUBLE)
RETURNS BOOLEAN
DETERMINISTIC
RETURN amount > 10000.0
-- Policy uses EXCEPT abac_tut_hr_team, abac_tut_admins to exempt those groups
""")

In [ ]:
spark.sql(f"""
-- Fraud detection policy: abac_tut_finance_team bypass the filter, others only see fraud=TRUE
CREATE OR REPLACE POLICY fraud_detection_filter
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.fraud_row_filter
TO `account users` EXCEPT abac_tut_finance_team, abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_emergency') AS fraud_col
USING COLUMNS (fraud_col)
""")


### What to expect from this query

`fraud_row_filter` returns `fraud_flag = TRUE`. Policy excepts finance_team, hr_team, and admins.

| Your Group | Transactions visible | Count |
|---|---|---|
| `abac_tut_finance_team` | All (fraud analysts — exempt) | 10 |
| `abac_tut_hr_team` | All (compliance — exempt) | 10 |
| `abac_tut_admins` | All (admin — exempt) | 10 |
| *(anyone else)* | Only fraud=TRUE: txns 1003, 1004, 1007, 1009 | 4 |


In [ ]:
print(f"Running as: {current_user}")
spark.sql(f"""SELECT txn_id, account_id, region, amount, fraud_flag, risk_score, txn_date FROM {catalog}.{schema}.transactions ORDER BY txn_date""").show(truncate=False)


**Expected results by role:**

| Role | Visible Transactions | Reason |
|------|---------------------|--------|
| `abac_tut_finance_team` | All 10 transactions | Exempt from fraud filter |
| `abac_tut_hr_team` | Txns 1002, 1004, 1006, 1008, 1010 (>$10K) | Exempt from fraud filter, see high-value |
| `abac_tut_admins` | All 10 transactions | Exempt from all filters |
| Standard user | Txns 1003, 1004, 1007, 1009 (fraud=TRUE) | Only fraud-flagged visible |